In [1]:
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import CreateTable, CreateTableColumn, CreateTableConstraints, CreateForeignKey
import pandas as pd
from pandas.core.interchange.dataframe_protocol import DataFrame
from dotenv import load_dotenv
import os 
from dbrepo.api.dto import CreateView
from dbrepo.api.dto import CreateView, Subset, SubsetColumn, Join
from dbrepo.api.dto import JoinType

load_dotenv()
password = os.getenv("DBREPO_PASS")
username = os.getenv("DBREPO_USER")
client = RestClient("https://test.dbrepo.tuwien.ac.at/", username=username, password=password)

containers = client.get_containers()
print(containers)


[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]


In [2]:
df = client.get_database("cf27a11d-58e5-4693-856c-e8f3527e3394")

# Create View 1: Summary of drugs in water

In [3]:
table = client.get_table("cf27a11d-58e5-4693-856c-e8f3527e3394", "c8dd8f35-ed00-4422-91a2-8c6b6c3a0fdd")
print(table)  # or the UUID of the table
for col in table.columns:
    print(col.id, col.name)

id='c8dd8f35-ed00-4422-91a2-8c6b6c3a0fdd' database_id='cf27a11d-58e5-4693-856c-e8f3527e3394' name='wastewater_data' owner=UserBrief(username='data_stewardship_group20', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) columns=[Column(id='4a112225-74ce-4151-84da-71ce05e95b4b', name='city_name', database_id='cf27a11d-58e5-4693-856c-e8f3527e3394', table_id='c8dd8f35-ed00-4422-91a2-8c6b6c3a0fdd', ord=0, internal_name='city_name', is_null_allowed=False, type=<ColumnType.VARCHAR: 'varchar'>, alias=None, description='The name of the city from EUDA/SCODA data (e.g., Graz)', size=100, d=None, mean=None, median=None, concept=None, unit=None, enums=[], sets=[], index_length=None, length=None, data_length=None, max_data_length=None, num_rows=None, val_min=None, val_max=None, std_dev=None), Column(id='52c4121c-e314-4df8-92b5-cb25cb1c473f', name='ref_year', database_id='cf27a11d-58e5-4693-856c-e8f3527e3394', table_id='c8dd8f35-ed00-4422-91a2-8c6b6c3a0fdd', ord=

In [4]:
df_city_summary_view = CreateView(
    name="vw_city_year_drug_summary",
    description="City-level aggregated wastewater indicators per year",
    is_public=True,
    is_schema_public=True,
    query=Subset(
        datasource_ids=["c8dd8f35-ed00-4422-91a2-8c6b6c3a0fdd"],  # wastewater_data table ID

        columns=[
            SubsetColumn(
                id="4a112225-74ce-4151-84da-71ce05e95b4b",
                alias="city_name"
            ),
            SubsetColumn(
                id="52c4121c-e314-4df8-92b5-cb25cb1c473f",
                alias="ref_year"
            ),

            SubsetColumn(
                id="2d3f5649-efc0-46a1-8bee-f9fe393cdbec",
                aggregation="avg",
                alias="avg_daily_mean"
            ),
            SubsetColumn(
                id="2d3f5649-efc0-46a1-8bee-f9fe393cdbec",
                aggregation="max",
                alias="max_daily_mean"
            ),

            SubsetColumn(
                id="9818c0d3-007c-4220-a2c3-5c55dffb8916",
                aggregation="count_distinct",
                alias="metabolite_count"
            )
        ],

        joins=None,
        filters=None,
        orders=None
    )
)


In [5]:
response = client._wrapper(
    method="post",
    url="/api/v1/database/cf27a11d-58e5-4693-856c-e8f3527e3394/view",
    payload=df_city_summary_view
)

print(response.status_code)
print(response.text)

401



# Create View 2: Join all

In [6]:
table = client.get_table("cf27a11d-58e5-4693-856c-e8f3527e3394", "64b402ae-b2e2-42dd-b1c8-9410131021ea")
print(table)  # or the UUID of the table
for col in table.columns:
    print(col.id, col.name)

id='64b402ae-b2e2-42dd-b1c8-9410131021ea' database_id='cf27a11d-58e5-4693-856c-e8f3527e3394' name='city_map' owner=UserBrief(username='data_stewardship_group20', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) columns=[Column(id='4447f084-0691-4492-ab66-8219b7142834', name='nuts_code', database_id='cf27a11d-58e5-4693-856c-e8f3527e3394', table_id='64b402ae-b2e2-42dd-b1c8-9410131021ea', ord=0, internal_name='nuts_code', is_null_allowed=False, type=<ColumnType.VARCHAR: 'varchar'>, alias=None, description='5-character NUTS-3 administrative code (e.g., AT221): https://ec.europa.eu/eurostat/web/nuts', size=5, d=None, mean=None, median=None, concept=None, unit=None, enums=[], sets=[], index_length=None, length=None, data_length=None, max_data_length=None, num_rows=None, val_min=None, val_max=None, std_dev=None), Column(id='34e908a3-bae5-47d1-a7dd-a40a9a538d98', name='city_name', database_id='cf27a11d-58e5-4693-856c-e8f3527e3394', table_id='64b402ae-b2e2

In [7]:
table = client.get_table("cf27a11d-58e5-4693-856c-e8f3527e3394", "319fcfe9-e536-4b0c-9be7-84c38f9cc346")
print(table) 
for col in table.columns:
    print(col.id, col.name)

id='319fcfe9-e536-4b0c-9be7-84c38f9cc346' database_id='cf27a11d-58e5-4693-856c-e8f3527e3394' name='gdp_data' owner=UserBrief(username='data_stewardship_group20', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) columns=[Column(id='e85bbc12-aac7-4738-be51-6e47a1e9b565', name='nuts_code', database_id='cf27a11d-58e5-4693-856c-e8f3527e3394', table_id='319fcfe9-e536-4b0c-9be7-84c38f9cc346', ord=0, internal_name='nuts_code', is_null_allowed=False, type=<ColumnType.VARCHAR: 'varchar'>, alias=None, description='5-character NUTS-3 administrative code (e.g., AT221): https://ec.europa.eu/eurostat/web/nuts', size=5, d=None, mean=None, median=None, concept=None, unit=None, enums=[], sets=[], index_length=None, length=None, data_length=None, max_data_length=None, num_rows=None, val_min=None, val_max=None, std_dev=None), Column(id='80ba5501-df7a-4587-8dfa-aa8575b866b0', name='city_name', database_id='cf27a11d-58e5-4693-856c-e8f3527e3394', table_id='319fcfe9-e536

In [9]:
df_ml_view = CreateView(
    name="drug_gdp_features_view",
    description="ML-ready dataset joining wastewater measurements with GDP via city-NUTS mapping",
    is_public=True,
    is_schema_public=True,
    query=Subset(
        datasource_ids=[
            "c8dd8f35-ed00-4422-91a2-8c6b6c3a0fdd"  # wastewater_data
        ],

        columns=[
            # wastewater_data
            SubsetColumn(
                id="4a112225-74ce-4151-84da-71ce05e95b4b",  # city_name
                alias="city_name"
            ),
            SubsetColumn(
                id="52c4121c-e314-4df8-92b5-cb25cb1c473f",  # ref_year
                alias="ref_year"
            ),
            SubsetColumn(
                id="9818c0d3-007c-4220-a2c3-5c55dffb8916",  # metabolite_name
                alias="metabolite_name"
            ),
            SubsetColumn(
                id="2d3f5649-efc0-46a1-8bee-f9fe393cdbec",  # daily_mean
                alias="daily_mean"
            ),

            # city_map.nuts_code
            SubsetColumn(
                id="4447f084-0691-4492-ab66-8219b7142834",  # nuts_code in city_map
                alias="nuts_code",
                join_alias="m"
            ),

            # gdp_data.gdp_per_cap
            SubsetColumn(
                id="76efa3db-c8d6-457d-bf5d-9280520a52ab",  # gdp_per_cap
                alias="gdp_per_cap",
                join_alias="g"
            )
        ],

        joins=[
            # wastewater_data → city_map
            Join(
                type=JoinType.INNER,
                datasource_id="64b402ae-b2e2-42dd-b1c8-9410131021ea",
                alias="m",
                conditionals=[
                    {
                        "column_id": "4a112225-74ce-4151-84da-71ce05e95b4b",        # wastewater.city_name
                        "foreign_column_id": "34e908a3-bae5-47d1-a7dd-a40a9a538d98" # city_map.city_name
                    }
                ]
            ),

            # city_map → gdp_data
            Join(
                type=JoinType.INNER,
                datasource_id="319fcfe9-e536-4b0c-9be7-84c38f9cc346",
                alias="g",
                conditionals=[
                    {
                        "column_id": "4447f084-0691-4492-ab66-8219b7142834",        # city_map.nuts_code
                        "foreign_column_id": "e85bbc12-aac7-4738-be51-6e47a1e9b565" # gdp.nuts_code
                    }
                ]
            )
                    ],

        filters=None,
        orders=None
    )
)



In [10]:
response = client._wrapper(
    method="post",
    url="/api/v1/database/cf27a11d-58e5-4693-856c-e8f3527e3394/view",
    payload=df_ml_view
)

print(response.status_code)
print(response.text)

401

